# Phase 1 — Tokenization & Text Preprocessing

**Duration:** Weeks 1–2 | **Difficulty:** ⭐☆☆☆☆

Welcome to the first phase of your NLP learning journey! This is the entry point to every NLP pipeline. Raw text is messy — it contains HTML tags, URLs, punctuation, inconsistent casing, extra spaces, and noise. Before any model can process text, you must clean it and split it into meaningful units called **tokens**. Every downstream phase depends on this phase being done correctly.

### Sequential Task Flow

```
Raw Text → Data Cleaning → Word Tokenization → Sentence Tokenization
        → Stopword Removal → Stemming → Lemmatization → Full Pipeline Comparison
```

## Table of Contents

- [0. Setup & Imports](#0-setup--imports)
- [Task 1 — Data Cleaning](#task-1--data-cleaning)
- [Task 2 — Word Tokenization](#task-2--word-tokenization)
- [Task 3 — Sentence Tokenization](#task-3--sentence-tokenization)
- [Task 4 — Stopword Removal](#task-4--stopword-removal)
- [Task 5 — Stemming](#task-5--stemming)
- [Task 6 — Lemmatization](#task-6--lemmatization)
- [Task 7 — Full Pipeline Comparison](#task-7--full-pipeline-comparison)
- [Summary & Key Takeaways](#summary--key-takeaways)

## 0. Setup & Imports

Before we begin, let us import all necessary libraries and download the required NLTK data packages. We install everything once at the top so every cell below can assume these are available.

In [1]:
import re
import unicodedata
import html as html_module

import nltk
import spacy
import pandas as pd
import numpy as np

# Download NLTK data packages (only needed once)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Load the spaCy English model
nlp = spacy.load('en_core_web_sm')

print('All libraries imported and data downloaded successfully.')
print(f'  NLTK version : {nltk.__version__}')
print(f'  spaCy version: {spacy.__version__}')
print(f'  Pandas version: {pd.__version__}')

All libraries imported and data downloaded successfully.
  NLTK version : 3.9.4
  spaCy version: 3.8.14
  Pandas version: 3.0.3


---
## Task 1 — Data Cleaning

**Objective:** Transform raw, noisy text into a clean, normalized string ready for tokenization.

Real-world text is rarely clean. It arrives with HTML markup, embedded URLs, inconsistent casing, punctuation noise, and erratic whitespace. Each of these must be handled before any NLP model can process the text reliably.

### Step 1 — Load raw text

> **What to do:** Collect 5–10 sentences of messy real-world text. Include at least one sentence with a URL, one with HTML tags, one with numbers, one with special characters, and one in ALL CAPS.
>
> **Display:** Print the raw text exactly as loaded — every piece of noise visible.
>
> **Infer:** Identify every type of noise present. This list becomes your cleaning checklist.

In [2]:
raw_sentences = [
    "Check out this AMAZING article at https://example.com/nlp-guide -- mind-blowing!!!",
    "<p>Natural Language Processing</p> is a <b>subfield</b> of AI &amp; linguistics.",
    "The product costs $1,299.99 and has a 4.5 star rating (out of 5).",
    "I    love   NLP...  it is    so    COOL!!!   Right?!?!",
    "BREAKING NEWS: Python 3.12 released -- performance improved by 25%!!!",
    "Email me at user@example.com or visit www.example.org for details.",
    "RT @nlp_fan: #MachineLearning is the future of #AI!! Check it out!!!"
]

print('=' * 80)
print('RAW TEXT (as loaded)')
print('=' * 80)
for i, sent in enumerate(raw_sentences, 1):
    print(f'  [{i}] {sent}')

print()
print('=' * 80)
print('NOISE INVENTORY')
print('=' * 80)
noise_types = [
    "URLs         : https://example.com/nlp-guide, www.example.org",
    "HTML tags    : <p>, </p>, <b>, </b>",
    "HTML entities: &amp;",
    "ALL CAPS     : AMAZING, COOL, BREAKING NEWS",
    "Special chars: $, @, #, %, !, ?",
    "Numbers      : $1,299.99, 4.5, 25%, 3.12",
    "Extra spaces : multiple consecutive spaces in sentence [4]",
    "Punctuation  : !!!, ?!?!, ...",
    "Email address: user@example.com",
    "Mentions/tags: @nlp_fan, #MachineLearning, #AI"
]
for n in noise_types:
    print(f'  {n}')

RAW TEXT (as loaded)
  [1] Check out this AMAZING article at https://example.com/nlp-guide -- mind-blowing!!!
  [2] <p>Natural Language Processing</p> is a <b>subfield</b> of AI &amp; linguistics.
  [3] The product costs $1,299.99 and has a 4.5 star rating (out of 5).
  [4] I    love   NLP...  it is    so    COOL!!!   Right?!?!
  [5] BREAKING NEWS: Python 3.12 released -- performance improved by 25%!!!
  [6] Email me at user@example.com or visit www.example.org for details.
  [7] RT @nlp_fan: #MachineLearning is the future of #AI!! Check it out!!!

NOISE INVENTORY
  URLs         : https://example.com/nlp-guide, www.example.org
  HTML tags    : <p>, </p>, <b>, </b>
  HTML entities: &amp;
  ALL CAPS     : AMAZING, COOL, BREAKING NEWS
  Special chars: $, @, #, %, !, ?
  Numbers      : $1,299.99, 4.5, 25%, 3.12
  Extra spaces : multiple consecutive spaces in sentence [4]
  Punctuation  : !!!, ?!?!, ...
  Email address: user@example.com
  Mentions/tags: @nlp_fan, #MachineLearning, #AI


### Step 2 — Lowercase the text

> **What to do:** Convert the entire text string to lowercase.
>
> **Display:** Print the original text and the lowercased version side by side.
>
> **Infer:** Observe that `NLP`, `nlp`, and `Nlp` now all look the same. Note that this is not always desirable — `US` (country) and `us` (pronoun) are now identical.

In [3]:
print(f"{'ORIGINAL':<55} | {'LOWERCASED'}")
print('-' * 120)
for sent in raw_sentences:
    lower = sent.lower()
    orig_display = sent[:52] + '...' if len(sent) > 55 else sent
    lower_display = lower[:52] + '...' if len(lower) > 55 else lower
    print(f'{orig_display:<55} | {lower_display}')

print()
print('LIMITATION: US (country) and us (pronoun) become identical after lowercasing.')
print('  NLP and nlp merge correctly, but IT (company) and it (pronoun) also merge.')
print('  Lowercasing is a lossy operation. It helps most tasks but hurts entity recognition.')

ORIGINAL                                                | LOWERCASED
------------------------------------------------------------------------------------------------------------------------
Check out this AMAZING article at https://example.co... | check out this amazing article at https://example.co...
<p>Natural Language Processing</p> is a <b>subfield<... | <p>natural language processing</p> is a <b>subfield<...
The product costs $1,299.99 and has a 4.5 star ratin... | the product costs $1,299.99 and has a 4.5 star ratin...
I    love   NLP...  it is    so    COOL!!!   Right?!?!  | i    love   nlp...  it is    so    cool!!!   right?!?!
BREAKING NEWS: Python 3.12 released -- performance i... | breaking news: python 3.12 released -- performance i...
Email me at user@example.com or visit www.example.or... | email me at user@example.com or visit www.example.or...
RT @nlp_fan: #MachineLearning is the future of #AI!!... | rt @nlp_fan: #machinelearning is the future of #ai!!...

LIMITATION: 

### Step 3 — Remove HTML tags

> **What to do:** Use a regex pattern to strip anything that looks like an HTML tag (text enclosed in `< >`).
>
> **Display:** Print a before/after pair for a sentence that contained an HTML tag.
>
> **Infer:** Confirm that the tag structure is gone but the text content between tags is preserved.

In [4]:
def remove_html_tags(text):
    """Remove HTML tags using regex and decode HTML entities."""
    text = re.sub(r'<[^>]+>', '', text)
    text = html_module.unescape(text)
    return text

html_sentence = raw_sentences[1]
cleaned = remove_html_tags(html_sentence)

print('BEFORE:', html_sentence)
print('AFTER: ', cleaned)
print()
print('HTML tags (<p>, </p>, <b>, </b>) removed.')
print('HTML entity &amp; decoded to &.')
print('Text content between tags preserved.')

BEFORE: <p>Natural Language Processing</p> is a <b>subfield</b> of AI &amp; linguistics.
AFTER:  Natural Language Processing is a subfield of AI & linguistics.

HTML tags (<p>, </p>, <b>, </b>) removed.
HTML entity &amp; decoded to &.
Text content between tags preserved.


### Step 4 — Remove URLs

> **What to do:** Use a regex pattern to remove `http://`, `https://`, and `www.` URLs entirely.
>
> **Display:** Print before/after for URL-containing sentences.
>
> **Infer:** URLs add no semantic meaning to most NLP tasks. Removing them reduces vocabulary noise.

In [5]:
def remove_urls(text):
    """Remove URLs starting with http://, https://, or www."""
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    return text

for idx in [0, 5]:
    original = raw_sentences[idx]
    cleaned = remove_urls(original)
    print(f'BEFORE: {original}')
    print(f'AFTER:  {cleaned.strip()}')
    print('-' * 80)

print()
print('URLs removed. They add no semantic meaning to most NLP tasks.')

BEFORE: Check out this AMAZING article at https://example.com/nlp-guide -- mind-blowing!!!
AFTER:  Check out this AMAZING article at  -- mind-blowing!!!
--------------------------------------------------------------------------------
BEFORE: Email me at user@example.com or visit www.example.org for details.
AFTER:  Email me at user@example.com or visit  for details.
--------------------------------------------------------------------------------

URLs removed. They add no semantic meaning to most NLP tasks.


### Step 5 — Remove special characters and punctuation

> **What to do:** Strip all characters that are not letters, digits, or spaces.
>
> **Display:** Print text before and after. Also print the unique characters removed.
>
> **Infer:** Punctuation like `!` and `?` carried sentiment signal. This step may discard useful signals — a tradeoff to revisit in Phase 4.

In [6]:
def remove_special_chars(text):
    """Remove all characters that are not letters, digits, or spaces."""
    removed = set(c for c in text if not c.isalnum() and not c.isspace())
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return cleaned, removed

test_sentences = [raw_sentences[0], raw_sentences[2], raw_sentences[3]]
all_removed = set()

for sent in test_sentences:
    cleaned, removed = remove_special_chars(sent)
    all_removed.update(removed)
    print(f'BEFORE: {sent}')
    print(f'AFTER:  {cleaned}')
    print(f'REMOVED: {sorted(removed)}')
    print('-' * 80)

print(f'All unique characters removed: {sorted(all_removed)}')
print(f'Total unique special characters: {len(all_removed)}')
print()
print('TRADEOFF: Punctuation like ! and ? carry sentiment signal.')
print('  AMAZING!!! has stronger sentiment than AMAZING.')
print('  We lose this signal. Revisit in Phase 4 (Sentiment Analysis).')

BEFORE: Check out this AMAZING article at https://example.com/nlp-guide -- mind-blowing!!!
AFTER:  Check out this AMAZING article at httpsexamplecomnlpguide  mindblowing
REMOVED: ['!', '-', '.', '/', ':']
--------------------------------------------------------------------------------
BEFORE: The product costs $1,299.99 and has a 4.5 star rating (out of 5).
AFTER:  The product costs 129999 and has a 45 star rating out of 5
REMOVED: ['$', '(', ')', ',', '.']
--------------------------------------------------------------------------------
BEFORE: I    love   NLP...  it is    so    COOL!!!   Right?!?!
AFTER:  I    love   NLP  it is    so    COOL   Right
REMOVED: ['!', '.', '?']
--------------------------------------------------------------------------------
All unique characters removed: ['!', '$', '(', ')', ',', '-', '.', '/', ':', '?']
Total unique special characters: 10

TRADEOFF: Punctuation like ! and ? carry sentiment signal.
  AMAZING!!! has stronger sentiment than AMAZING.
  We lo

### Step 6 — Remove extra whitespace

> **What to do:** Collapse multiple spaces, tabs, and newlines into a single space. Strip leading/trailing whitespace.
>
> **Display:** Print a final cleaned version of all sentences.
>
> **Infer:** The text should now be clean, lowercase, and uniform. This is what enters the tokenizer.

In [7]:
def full_clean(text):
    """Apply the complete cleaning pipeline."""
    text = text.lower()
    text = remove_html_tags(text)
    text = remove_urls(text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('=' * 100)
print(f"{'RAW TEXT':<60} | CLEANED TEXT")
print('=' * 100)
for sent in raw_sentences:
    cleaned = full_clean(sent)
    orig_display = sent[:57] + '...' if len(sent) > 60 else sent
    print(f'{orig_display:<60} | {cleaned}')

print()
print('All text is now lowercase, tag-free, URL-free, and uniformly spaced.')
print('This cleaned text is ready to enter the tokenizer in Task 2.')

RAW TEXT                                                     | CLEANED TEXT
Check out this AMAZING article at https://example.com/nlp... | check out this amazing article at mindblowing
<p>Natural Language Processing</p> is a <b>subfield</b> o... | natural language processing is a subfield of ai linguistics
The product costs $1,299.99 and has a 4.5 star rating (ou... | the product costs 129999 and has a 45 star rating out of 5
I    love   NLP...  it is    so    COOL!!!   Right?!?!       | i love nlp it is so cool right
BREAKING NEWS: Python 3.12 released -- performance improv... | breaking news python 312 released performance improved by 25
Email me at user@example.com or visit www.example.org for... | email me at usercom or visit for details
RT @nlp_fan: #MachineLearning is the future of #AI!! Chec... | rt machinelearning is the future of ai check it out

All text is now lowercase, tag-free, URL-free, and uniformly spaced.
This cleaned text is ready to enter the tokenizer in Task 2.


---
## Task 2 — Word Tokenization

**Objective:** Split a cleaned sentence into a list of individual word tokens.

Tokenization seems trivial — just split on spaces, right? This task reveals why it is not. Contractions, hyphenated compounds, and edge cases all demand different strategies. We compare three approaches: a naive whitespace split, NLTK rule-based tokenizer, and spaCy statistical tokenizer.

### Step 1 — Whitespace splitting (scratch)

> **What to do:** Split the cleaned sentence using Python built-in `.split()` method.
>
> **Display:** Print the resulting list of tokens and its length.
>
> **Infer:** This works for clean text. Test it on tricky cases to see where it fails.

In [8]:
clean_text = 'natural language processing is a subfield of artificial intelligence'
ws_tokens = clean_text.split()

print(f'Input:  {clean_text!r}')
print(f'Tokens: {ws_tokens}')
print(f'Count:  {len(ws_tokens)}')

tricky = "it's a state-of-the-art model"
tricky_tokens = tricky.split()

print(f'\nInput:  {tricky!r}')
print(f'Tokens: {tricky_tokens}')
print(f'Count:  {len(tricky_tokens)}')
print()
print('Observation:')
print('  it\'s stays as one token -- should it and \'s be separate?')
print('  state-of-the-art stays as one token -- is this one word or four?')
print('  Whitespace splitting has no opinion on these. It only sees spaces.')

Input:  'natural language processing is a subfield of artificial intelligence'
Tokens: ['natural', 'language', 'processing', 'is', 'a', 'subfield', 'of', 'artificial', 'intelligence']
Count:  9

Input:  "it's a state-of-the-art model"
Tokens: ["it's", 'a', 'state-of-the-art', 'model']
Count:  4

Observation:
  it's stays as one token -- should it and 's be separate?
  state-of-the-art stays as one token -- is this one word or four?
  Whitespace splitting has no opinion on these. It only sees spaces.


### Step 2 — NLTK word_tokenize

> **What to do:** Apply NLTK `word_tokenize` to the same sentences.
>
> **Display:** Print the token list side by side with the whitespace version.
>
> **Infer:** NLTK correctly splits contractions into separate tokens (Penn Treebank style).

In [9]:
from nltk.tokenize import word_tokenize

examples = [
    "it's a state-of-the-art model",
    "NLP is state-of-the-art! However, it doesn't always work perfectly.",
    "I can't believe it's not butter!",
]

for sent in examples:
    ws = sent.split()
    nk = word_tokenize(sent)
    print(f'Input: {sent!r}')
    print(f'  Whitespace ({len(ws):>2} tokens): {ws}')
    print(f'  NLTK       ({len(nk):>2} tokens): {nk}')
    print()

print('NLTK splits contractions: doesn\'t -> [does, n\'t], can\'t -> [ca, n\'t]')
print('This is Penn Treebank style -- it isolates the negation particle n\'t.')

Input: "it's a state-of-the-art model"
  Whitespace ( 4 tokens): ["it's", 'a', 'state-of-the-art', 'model']
  NLTK       ( 5 tokens): ['it', "'s", 'a', 'state-of-the-art', 'model']

Input: "NLP is state-of-the-art! However, it doesn't always work perfectly."
  Whitespace ( 9 tokens): ['NLP', 'is', 'state-of-the-art!', 'However,', 'it', "doesn't", 'always', 'work', 'perfectly.']
  NLTK       (13 tokens): ['NLP', 'is', 'state-of-the-art', '!', 'However', ',', 'it', 'does', "n't", 'always', 'work', 'perfectly', '.']

Input: "I can't believe it's not butter!"
  Whitespace ( 6 tokens): ['I', "can't", 'believe', "it's", 'not', 'butter!']
  NLTK       ( 9 tokens): ['I', 'ca', "n't", 'believe', 'it', "'s", 'not', 'butter', '!']

NLTK splits contractions: doesn't -> [does, n't], can't -> [ca, n't]
This is Penn Treebank style -- it isolates the negation particle n't.


### Step 3 — spaCy tokenizer

> **What to do:** Process the same sentences using spaCy and extract tokens.
>
> **Display:** Print the token list with `.is_punct` and `.is_space` attributes.
>
> **Infer:** Compare spaCy output to NLTK. Note which tokenizer handles each case better.

In [10]:
test_sent = "NLP is state-of-the-art! However, it doesn't always work perfectly."
doc = nlp(test_sent)

print(f'Input: {test_sent!r}')
print()
print(f'{"Token":<20} | {"is_punct":<10} | {"is_space":<10} | POS')
print('-' * 65)
for token in doc:
    print(f'{token.text:<20} | {str(token.is_punct):<10} | {str(token.is_space):<10} | {token.pos_}')

spacy_tokens = [token.text for token in doc]
nltk_tokens = word_tokenize(test_sent)

print(f'\nspaCy tokens ({len(spacy_tokens)}): {spacy_tokens}')
print(f'NLTK  tokens ({len(nltk_tokens)}): {nltk_tokens}')
print()
print('Key differences:')
print('  spaCy splits state-of-the-art into 7 tokens (word-hyphen-word...)')
print('  NLTK keeps it as one token.')
print('  Both split contractions the same way.')
print('  spaCy also provides POS tags alongside tokenization.')

Input: "NLP is state-of-the-art! However, it doesn't always work perfectly."

Token                | is_punct   | is_space   | POS
-----------------------------------------------------------------
NLP                  | False      | False      | PROPN
is                   | False      | False      | AUX
state                | False      | False      | NOUN
-                    | True       | False      | PUNCT
of                   | False      | False      | ADP
-                    | True       | False      | PUNCT
the                  | False      | False      | DET
-                    | True       | False      | PUNCT
art                  | False      | False      | NOUN
!                    | True       | False      | PUNCT
However              | False      | False      | ADV
,                    | True       | False      | PUNCT
it                   | False      | False      | PRON
does                 | False      | False      | AUX
n't                  | False      | False     

### Step 4 — Edge case comparison

> **What to do:** Run all three tokenizers on deliberate edge cases.
>
> **Display:** Print a table with the input and each tokenizer output.
>
> **Infer:** No single tokenizer handles every edge case. Production pipelines often pre-clean text.

In [11]:
edge_cases = [
    "Visit https://example.com for details.",
    "Contact user@example.com for help.",
    "The cost is $1,000.00 exactly.",
    "Wait for it... the results are amazing...",
    "The U.S.A. is a country."
]

print(f'{"Edge Case":<50} | {"WS":>3} | {"NLTK":>4} | {"spaCy":>5}')
print('=' * 75)

for sent in edge_cases:
    ws = len(sent.split())
    nk = len(word_tokenize(sent))
    sp = len([t for t in nlp(sent)])
    print(f'{sent:<50} | {ws:>3} | {nk:>4} | {sp:>5}')

print()
print('No single tokenizer handles every edge case perfectly.')
print('Production pipelines often pre-clean text before tokenizing.')

Edge Case                                          |  WS | NLTK | spaCy
Visit https://example.com for details.             |   4 |    7 |     5
Contact user@example.com for help.                 |   4 |    7 |     5
The cost is $1,000.00 exactly.                     |   5 |    7 |     7
Wait for it... the results are amazing...          |   7 |    9 |     9
The U.S.A. is a country.                           |   5 |    6 |     6

No single tokenizer handles every edge case perfectly.
Production pipelines often pre-clean text before tokenizing.


---
## Task 3 — Sentence Tokenization

**Objective:** Split a paragraph or document into individual sentences.

Sentence tokenization seems simple — split on periods. But abbreviations like `Dr.` and `U.S.A.` break naive period splitting. This task compares a scratch approach against NLTK trained Punkt tokenizer.

### Step 1 — Period splitting (scratch)

> **What to do:** Split a paragraph on the `.` character.
>
> **Display:** Print the resulting sentence list.
>
> **Infer:** Abbreviations like `Dr.` and `U.S.A.` break the simple period split. List every wrong split.

In [12]:
paragraph = (
    "Dr. Smith bought a laptop for $1,299.99. "
    "It was manufactured in the U.S.A. by a well-known company. "
    "He was very happy with the purchase! "
    "The delivery took approx. 3 days. What a deal!"
)

naive_sentences = [s.strip() for s in paragraph.split('.') if s.strip()]

print('Input paragraph:')
print(f'  {paragraph!r}')
print(f'\nNaive period split ({len(naive_sentences)} fragments):')
print('-' * 60)
for i, s in enumerate(naive_sentences, 1):
    print(f'  [{i}] {s}')

print()
print('PROBLEMS:')
print('  Dr. caused a false split')
print('  $1,299.99 caused a split mid-number')
print('  U.S.A. caused multiple false splits')
print('  approx. caused a false split')
print('  Period splitting fails on abbreviations and decimal numbers.')

Input paragraph:
  'Dr. Smith bought a laptop for $1,299.99. It was manufactured in the U.S.A. by a well-known company. He was very happy with the purchase! The delivery took approx. 3 days. What a deal!'

Naive period split (10 fragments):
------------------------------------------------------------
  [1] Dr
  [2] Smith bought a laptop for $1,299
  [3] 99
  [4] It was manufactured in the U
  [5] S
  [6] A
  [7] by a well-known company
  [8] He was very happy with the purchase! The delivery took approx
  [9] 3 days
  [10] What a deal!

PROBLEMS:
  Dr. caused a false split
  $1,299.99 caused a split mid-number
  U.S.A. caused multiple false splits
  approx. caused a false split
  Period splitting fails on abbreviations and decimal numbers.


### Step 2 — NLTK sent_tokenize

> **What to do:** Apply NLTK `sent_tokenize` on the same paragraph.
>
> **Display:** Print the sentence list with index numbers and count.
>
> **Infer:** NLTK Punkt tokenizer handles abbreviations much better.

In [13]:
from nltk.tokenize import sent_tokenize

nltk_sentences = sent_tokenize(paragraph)
spacy_sentences = [sent.text.strip() for sent in nlp(paragraph).sents]

print(f'NLTK sent_tokenize ({len(nltk_sentences)} sentences):')
print('-' * 60)
for i, s in enumerate(nltk_sentences, 1):
    print(f'  [{i}] {s}')

print(f'\nspaCy sentence segmenter ({len(spacy_sentences)} sentences):')
print('-' * 60)
for i, s in enumerate(spacy_sentences, 1):
    print(f'  [{i}] {s}')

print()
print('NLTK Punkt tokenizer correctly handles Dr. as an abbreviation.')
print('spaCy uses its dependency parser to detect sentence boundaries.')

NLTK sent_tokenize (6 sentences):
------------------------------------------------------------
  [1] Dr. Smith bought a laptop for $1,299.99.
  [2] It was manufactured in the U.S.A. by a well-known company.
  [3] He was very happy with the purchase!
  [4] The delivery took approx.
  [5] 3 days.
  [6] What a deal!

spaCy sentence segmenter (6 sentences):
------------------------------------------------------------
  [1] Dr. Smith bought a laptop for $1,299.99.
  [2] It was manufactured in the U.S.A. by a well-known company.
  [3] He was very happy with the purchase!
  [4] The delivery took approx.
  [5] 3 days.
  [6] What a deal!

NLTK Punkt tokenizer correctly handles Dr. as an abbreviation.
spaCy uses its dependency parser to detect sentence boundaries.


### Step 3 — Longer document comparison

> **What to do:** Take a multi-paragraph text. Compare naive vs trained methods.
>
> **Display:** Print sentence count from each method.
>
> **Infer:** Quantify the error rate of naive period-split. This motivates trained tokenizers.

In [14]:
long_text = (
    "The company, founded by Dr. J. Smith in 2019, is based in Washington, D.C. "
    "It specializes in A.I. and machine learning applications. "
    "Their latest product costs $4,999.99 and ships to the U.S., U.K., and E.U. "
    "Rev. Johnson praised the innovation at the annual tech conference. "
    "The stock price rose by 15.7% in Q3. Analysts predict continued growth. "
    "For more info, visit the company website. Contact support at ext. 4521."
)

naive_count = len([s for s in long_text.split('.') if s.strip()])
nltk_count = len(sent_tokenize(long_text))
spacy_count = len(list(nlp(long_text).sents))

print('Sentence counts:')
print(f'  Naive period split: {naive_count} fragments')
print(f'  NLTK sent_tokenize: {nltk_count} sentences')
print(f'  spaCy segmenter:    {spacy_count} sentences')
print(f'\nNaive method over-counts by {naive_count - nltk_count} fragments')
print('This is why production systems use trained tokenizers.')

Sentence counts:
  Naive period split: 22 fragments
  NLTK sent_tokenize: 11 sentences
  spaCy segmenter:    7 sentences

Naive method over-counts by 11 fragments
This is why production systems use trained tokenizers.


---
## Task 4 — Stopword Removal

**Objective:** Filter out high-frequency, low-information words that add noise to text representations.

Stopwords like `the`, `is`, `and`, `a` appear in almost every sentence but carry little semantic meaning. Removing them reduces vocabulary size. However, this section demonstrates a critical failure: removing `not` can destroy meaning.

### Step 1 — Inspect the stopword list

> **What to do:** Load NLTK English stopword list and print it.
>
> **Display:** Print all stopwords sorted alphabetically with total count.
>
> **Infer:** Notice it includes `not`, `no`, `nor`. Removing these eliminates negation signals.

In [15]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
sorted_stops = sorted(stop_words)

print(f'NLTK English Stopwords ({len(stop_words)} words):')
print('-' * 80)

cols = 6
for i in range(0, len(sorted_stops), cols):
    row = sorted_stops[i:i+cols]
    print('  ' + '  '.join(f'{w:<12}' for w in row))

# Highlight dangerous stopwords
danger_words = sorted([w for w in stop_words if w in {'not', 'no', 'nor', 'don', "don't",
    'doesn', "doesn't", 'didn', "didn't", 'won', "won't", 'wouldn', "wouldn't",
    'shouldn', "shouldn't", 'couldn', "couldn't", 'isn', "isn't", 'aren', "aren't",
    'wasn', "wasn't", 'weren', "weren't", 'hasn', "hasn't", 'haven', "haven't",
    'hadn', "hadn't"}])

print(f'\nDANGER: The following negation words are in the stopword list:')
print(f'  {danger_words}')
print('  Removing these will flip the meaning of negative sentences!')

NLTK English Stopwords (198 words):
--------------------------------------------------------------------------------
  a             about         above         after         again         against     
  ain           all           am            an            and           any         
  are           aren          aren't        as            at            be          
  because       been          before        being         below         between     
  both          but           by            can           couldn        couldn't    
  d             did           didn          didn't        do            does        
  doesn         doesn't       doing         don           don't         down        
  during        each          few           for           from          further     
  had           hadn          hadn't        has           hasn          hasn't      
  have          haven         haven't       having        he            he'd        
  he'll         he's          her

### Step 2 — Remove stopwords from a sentence

> **What to do:** Filter out tokens that appear in the stopword set.
>
> **Display:** Print original tokens, filtered tokens, and removed tokens.
>
> **Infer:** Count the reduction. The remaining words are the content-bearing ones.

In [16]:
sample = 'The quick brown fox jumps over the lazy dog in the park'
tokens = word_tokenize(sample.lower())
filtered = [t for t in tokens if t not in stop_words]
removed = [t for t in tokens if t in stop_words]

print(f'Original:  {sample!r}')
print(f'Tokens:    {tokens} ({len(tokens)} tokens)')
print(f'Filtered:  {filtered} ({len(filtered)} tokens)')
print(f'Removed:   {removed} ({len(removed)} tokens)')
print(f'\nReduction: {len(tokens)} -> {len(filtered)} tokens ({len(removed)} removed, {len(removed)/len(tokens)*100:.0f}% reduction)')
print(f'\nThe remaining words -- {filtered} -- carry the core meaning.')

Original:  'The quick brown fox jumps over the lazy dog in the park'
Tokens:    ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'in', 'the', 'park'] (12 tokens)
Filtered:  ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog', 'park'] (7 tokens)
Removed:   ['the', 'over', 'the', 'in', 'the'] (5 tokens)

Reduction: 12 -> 7 tokens (5 removed, 42% reduction)

The remaining words -- ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog', 'park'] -- carry the core meaning.


### Step 3 — Negation failure demo

> **What to do:** Run stopword removal on opposite sentences.
>
> **Display:** Print both outputs after stopword removal.
>
> **Infer:** Both produce nearly identical outputs despite opposite meanings. This is a critical failure mode.

In [17]:
positive = 'this movie is good'
negative = 'this movie is not good at all'

pos_tokens = word_tokenize(positive.lower())
neg_tokens = word_tokenize(negative.lower())

pos_filtered = [t for t in pos_tokens if t not in stop_words]
neg_filtered = [t for t in neg_tokens if t not in stop_words]

print('NEGATION FAILURE DEMONSTRATION')
print('=' * 60)
print(f'Positive: {positive!r}')
print(f'  After stopword removal: {pos_filtered}')
print()
print(f'Negative: {negative!r}')
print(f'  After stopword removal: {neg_filtered}')
print()
print(f'CRITICAL: Both sentences produce the SAME output: {pos_filtered}')
print('Yet they have OPPOSITE meanings!')
print('The word not was removed because it is in the default stopword list.')
print('Default stopword lists are DANGEROUS for sentiment analysis tasks.')

NEGATION FAILURE DEMONSTRATION
Positive: 'this movie is good'
  After stopword removal: ['movie', 'good']

Negative: 'this movie is not good at all'
  After stopword removal: ['movie', 'good']

CRITICAL: Both sentences produce the SAME output: ['movie', 'good']
Yet they have OPPOSITE meanings!
The word not was removed because it is in the default stopword list.
Default stopword lists are DANGEROUS for sentiment analysis tasks.


### Step 4 — Custom stopword list

> **What to do:** Add domain-specific words, remove `not` from the default list.
>
> **Display:** Show custom changes. Apply to sample sentences.
>
> **Infer:** Stopword lists should always be domain-tailored.

In [18]:
# Create a custom stopword list for movie review analysis
custom_stops = stop_words.copy()

# REMOVE negation words -- we want to preserve them
negation_words = {'not', 'no', 'nor', 'never', 'neither'}
custom_stops -= negation_words

# ADD domain-specific low-information words
domain_stops = {'movie', 'film', 'watch', 'scene', 'character', 'story', 'plot'}
custom_stops |= domain_stops

print(f'Default stopword count: {len(stop_words)}')
print(f'Custom stopword count:  {len(custom_stops)}')
print(f'Negation words PRESERVED: {sorted(negation_words & stop_words)}')
print(f'Domain words ADDED:       {sorted(domain_stops)}')

review_sentences = [
    "this movie is not good at all",
    "the film had a great story and wonderful character development",
    "I would never watch this movie again it was terrible",
    "the plot was not bad but the scene transitions were awful",
    "an absolutely brilliant film with stunning performances"
]

print()
print('=' * 80)
print(f'{"SENTENCE":<55} | CUSTOM FILTERED')
print('=' * 80)
for sent in review_sentences:
    tokens = word_tokenize(sent.lower())
    filtered = [t for t in tokens if t not in custom_stops]
    print(f'{sent:<55} | {filtered}')

print()
print('Now not is preserved: not good keeps its negative meaning.')
print('Domain words like movie, film, story are removed as noise.')
print('Always customize stopword lists for your specific task and domain.')

Default stopword count: 198
Custom stopword count:  202
Negation words PRESERVED: ['no', 'nor', 'not']
Domain words ADDED:       ['character', 'film', 'movie', 'plot', 'scene', 'story', 'watch']

SENTENCE                                                | CUSTOM FILTERED
this movie is not good at all                           | ['not', 'good']
the film had a great story and wonderful character development | ['great', 'wonderful', 'development']
I would never watch this movie again it was terrible    | ['would', 'never', 'terrible']
the plot was not bad but the scene transitions were awful | ['not', 'bad', 'transitions', 'awful']
an absolutely brilliant film with stunning performances | ['absolutely', 'brilliant', 'stunning', 'performances']

Now not is preserved: not good keeps its negative meaning.
Domain words like movie, film, story are removed as noise.
Always customize stopword lists for your specific task and domain.


---
## Task 5 — Stemming

**Objective:** Reduce words to their approximate root form by removing suffixes.

Stemming is a crude but fast heuristic: it chops off word endings using pattern rules. The result is often not a real English word (e.g., `studies` → `studi`), but it groups related word forms under a common root. We compare **Porter** (1980) and **Snowball** (improved version).

### Step 1 — Porter Stemmer

> **What to do:** Run Porter Stemmer on a diverse list of 20 words.
>
> **Display:** Print a two-column table: original word | stem.
>
> **Infer:** Count how many stems are not valid English words (over-stemming).

In [19]:
from nltk.stem import PorterStemmer

porter = PorterStemmer()

words = [
    "connecting", "connections", "connected", "connection",
    "studies", "studying", "studied", "study",
    "running", "runner", "ran", "runs",
    "better", "best", "good",
    "mice", "wolves", "children",
    "organization", "organize"
]

known_non_words = {'studi', 'wolv', 'organiz', 'childr'}

print(f'{"Original":<18} | {"Porter Stem":<15} | Real Word?')
print('-' * 55)

non_word_count = 0
for w in words:
    stem = porter.stem(w)
    is_non_word = stem in known_non_words
    if is_non_word:
        non_word_count += 1
    marker = 'NOT a word' if is_non_word else 'ok'
    print(f'{w:<18} | {stem:<15} | {marker}')

print(f'\n{non_word_count} out of {len(words)} stems are not valid English words.')
print('This is called OVER-STEMMING -- the stemmer cut off too much.')

Original           | Porter Stem     | Real Word?
-------------------------------------------------------
connecting         | connect         | ok
connections        | connect         | ok
connected          | connect         | ok
connection         | connect         | ok
studies            | studi           | NOT a word
studying           | studi           | NOT a word
studied            | studi           | NOT a word
study              | studi           | NOT a word
running            | run             | ok
runner             | runner          | ok
ran                | ran             | ok
runs               | run             | ok
better             | better          | ok
best               | best            | ok
good               | good            | ok
mice               | mice            | ok
wolves             | wolv            | NOT a word
children           | children        | ok
organization       | organ           | ok
organize           | organ           | ok

5 out of 20 s

### Step 2 — Snowball Stemmer comparison

> **What to do:** Run Snowball Stemmer on the same 20 words.
>
> **Display:** Add a third column to the table.
>
> **Infer:** Snowball is more conservative. Identify where they disagree.

In [20]:
from nltk.stem import SnowballStemmer

snowball = SnowballStemmer('english')

print(f'{"Original":<18} | {"Porter":<15} | {"Snowball":<15} | Match?')
print('-' * 70)

disagreements = 0
for w in words:
    p = porter.stem(w)
    s = snowball.stem(w)
    match = 'YES' if p == s else 'DIFFER'
    if p != s:
        disagreements += 1
    print(f'{w:<18} | {p:<15} | {s:<15} | {match}')

print(f'\nDisagreements: {disagreements} out of {len(words)} words')
print()
print('Key observations:')
print('  Both use suffix stripping rules')
print('  Snowball is sometimes more conservative')
print('  Neither understands meaning -- they only strip character patterns')
print('  Irregular forms (mice, ran, better) are NOT handled by either')

Original           | Porter          | Snowball        | Match?
----------------------------------------------------------------------
connecting         | connect         | connect         | YES
connections        | connect         | connect         | YES
connected          | connect         | connect         | YES
connection         | connect         | connect         | YES
studies            | studi           | studi           | YES
studying           | studi           | studi           | YES
studied            | studi           | studi           | YES
study              | studi           | studi           | YES
running            | run             | run             | YES
runner             | runner          | runner          | YES
ran                | ran             | ran             | YES
runs               | run             | run             | YES
better             | better          | better          | YES
best               | best            | best            | YES
good       

---
## Task 6 — Lemmatization

**Objective:** Reduce words to their true dictionary base form using linguistic analysis.

Unlike stemming, lemmatization uses vocabulary lookup and morphological analysis to return real dictionary words. `studies` → `study`, `better` → `good`, `mice` → `mouse`. The tradeoff: lemmatization is slower because it requires POS tagging and dictionary access.

### Step 1 — spaCy lemmatizer

> **What to do:** Process the same 20 words using spaCy. Extract `.lemma_`.
>
> **Display:** Add a fourth column: lemma. Compare to stems.
>
> **Infer:** Lemmas are always real dictionary words. `studies` → `study`, `mice` → `mouse`.

In [21]:
print(f'{"Original":<18} | {"Porter":<12} | {"Snowball":<12} | {"spaCy Lemma":<15} | Notes')
print('-' * 85)

for w in words:
    p = porter.stem(w)
    s = snowball.stem(w)
    doc = nlp(w)
    lemma = doc[0].lemma_

    notes = ''
    if lemma != p and lemma != s:
        notes = '<-- lemma differs from both stems'

    print(f'{w:<18} | {p:<12} | {s:<12} | {lemma:<15} | {notes}')

print()
print('Key observations:')
print('  Lemmas are always real dictionary words (unlike stems like studi, wolv)')
print('  mice -> mouse, wolves -> wolf, children -> child (handles irregular plurals)')
print('  Lemmatization is slower -- it needs POS tagging + dictionary lookup')

Original           | Porter       | Snowball     | spaCy Lemma     | Notes
-------------------------------------------------------------------------------------
connecting         | connect      | connect      | connect         | 
connections        | connect      | connect      | connection      | <-- lemma differs from both stems
connected          | connect      | connect      | connect         | 
connection         | connect      | connect      | connection      | <-- lemma differs from both stems
studies            | studi        | studi        | study           | <-- lemma differs from both stems
studying           | studi        | studi        | study           | <-- lemma differs from both stems
studied            | studi        | studi        | study           | <-- lemma differs from both stems
study              | studi        | studi        | study           | <-- lemma differs from both stems
running            | run          | run          | run             | 
runner     

wolves             | wolv         | wolv         | wolf            | <-- lemma differs from both stems
children           | children     | children     | child           | <-- lemma differs from both stems
organization       | organ        | organ        | organization    | <-- lemma differs from both stems
organize           | organ        | organ        | organize        | <-- lemma differs from both stems

Key observations:
  Lemmas are always real dictionary words (unlike stems like studi, wolv)
  mice -> mouse, wolves -> wolf, children -> child (handles irregular plurals)
  Lemmatization is slower -- it needs POS tagging + dictionary lookup


### Step 2 — POS dependency demo

> **What to do:** Run spaCy on the word `lead` in two different sentences.
>
> **Display:** Print the lemma and POS for `lead` in each sentence.
>
> **Infer:** Same word, different POS → potentially different lemma. Lemmatization requires context.

In [22]:
pos_sentences = [
    ("He will lead the team", "lead"),
    ("The pipe is made of lead", "lead"),
    ("I saw a bird in the garden", "saw"),
    ("He used a saw to cut the wood", "saw"),
]

print('POS DEPENDENCY DEMONSTRATION')
print('=' * 70)

for sent, target_word in pos_sentences:
    doc = nlp(sent)
    for token in doc:
        if token.text.lower() == target_word:
            pos_full = spacy.explain(token.pos_)
            print(f'\nSentence: {sent!r}')
            print(f'  Token: {token.text!r} -> POS: {token.pos_} ({pos_full}) -> Lemma: {token.lemma_!r}')

print()
print('Same word, different POS -> different lemma.')
print('  saw as VERB -> lemma see')
print('  saw as NOUN -> lemma saw')
print('Lemmatization REQUIRES sentence context. Stemming does not.')

POS DEPENDENCY DEMONSTRATION



Sentence: 'He will lead the team'
  Token: 'lead' -> POS: VERB (verb) -> Lemma: 'lead'

Sentence: 'The pipe is made of lead'
  Token: 'lead' -> POS: NOUN (noun) -> Lemma: 'lead'

Sentence: 'I saw a bird in the garden'
  Token: 'saw' -> POS: VERB (verb) -> Lemma: 'see'

Sentence: 'He used a saw to cut the wood'
  Token: 'saw' -> POS: NOUN (noun) -> Lemma: 'saw'

Same word, different POS -> different lemma.
  saw as VERB -> lemma see
  saw as NOUN -> lemma saw
Lemmatization REQUIRES sentence context. Stemming does not.


---
## Task 7 — Full Pipeline Comparison

**Objective:** See the complete preprocessing pipeline in one view — from raw text through every transformation stage.

This is the culmination of Tasks 1–6. We take 3 sentences through every step: raw text → clean → tokenize → remove stopwords → stem → lemmatize.

### Step 1 — Run the full pipeline

> **What to do:** Take 3 sentences through every step.
>
> **Display:** Print the text at each stage. Format as a pipeline view.
>
> **Infer:** At which stage was information lost? At which stage was noise removed? The answer depends on the task.

In [23]:
pipeline_sentences = [
    "The researchers at MIT published AMAZING results on NLP!!! Visit https://mit.edu for details.",
    "Dr. Smith's studies showed that running daily produces SIGNIFICANTLY better results than walking.",
    "<b>BREAKING</b>: The wolves & mice were not observed in the U.S.A. national parks this year."
]

for idx, raw in enumerate(pipeline_sentences, 1):
    print(f'\n{"=" * 90}')
    print(f'SENTENCE {idx}')
    print(f'{"=" * 90}')

    # Stage 1: Raw
    print(f'\n  1. RAW TEXT:')
    print(f'     {raw}')

    # Stage 2: Cleaned
    cleaned = full_clean(raw)
    print(f'\n  2. CLEANED (lowercase + remove HTML/URLs/special chars + normalize whitespace):')
    print(f'     {cleaned}')

    # Stage 3: Tokenized
    tokens = word_tokenize(cleaned)
    print(f'\n  3. TOKENIZED ({len(tokens)} tokens):')
    print(f'     {tokens}')

    # Stage 4: Stopwords removed
    filtered = [t for t in tokens if t not in stop_words]
    removed_sw = [t for t in tokens if t in stop_words]
    print(f'\n  4. STOPWORDS REMOVED ({len(filtered)} tokens, {len(removed_sw)} removed):')
    print(f'     Kept:    {filtered}')
    print(f'     Removed: {removed_sw}')

    # Stage 5: Stemmed
    stemmed = [porter.stem(t) for t in filtered]
    print(f'\n  5. STEMMED (Porter):')
    print(f'     {stemmed}')

    # Stage 6: Lemmatized
    doc = nlp(' '.join(filtered))
    lemmatized = [token.lemma_ for token in doc]
    print(f'\n  6. LEMMATIZED (spaCy):')
    print(f'     {lemmatized}')

print(f'\n{"=" * 90}')
print('OBSERVATIONS:')
print('=' * 90)
print('  Stage 2 (Cleaning): Noise removed, but ALL CAPS emphasis (AMAZING) lost')
print('  Stage 4 (Stopwords): not removed from sentence 3 -- negation lost!')
print('  Stage 5 (Stemming): Some stems are non-words (signific, studi)')
print('  Stage 6 (Lemmatization): All outputs are real words')
print('  Information loss depends on the task:')
print('    For classification: cleaning + stopword removal helps')
print('    For sentiment: removing not is catastrophic')
print('    For search/IR: stemming groups more aggressively than lemmatization')


SENTENCE 1

  1. RAW TEXT:
     The researchers at MIT published AMAZING results on NLP!!! Visit https://mit.edu for details.

  2. CLEANED (lowercase + remove HTML/URLs/special chars + normalize whitespace):
     the researchers at mit published amazing results on nlp visit for details

  3. TOKENIZED (12 tokens):
     ['the', 'researchers', 'at', 'mit', 'published', 'amazing', 'results', 'on', 'nlp', 'visit', 'for', 'details']

  4. STOPWORDS REMOVED (8 tokens, 4 removed):
     Kept:    ['researchers', 'mit', 'published', 'amazing', 'results', 'nlp', 'visit', 'details']
     Removed: ['the', 'at', 'on', 'for']

  5. STEMMED (Porter):
     ['research', 'mit', 'publish', 'amaz', 'result', 'nlp', 'visit', 'detail']

  6. LEMMATIZED (spaCy):
     ['researcher', 'mit', 'publish', 'amazing', 'result', 'nlp', 'visit', 'detail']

SENTENCE 2

  1. RAW TEXT:
     Dr. Smith's studies showed that running daily produces SIGNIFICANTLY better results than walking.

  2. CLEANED (lowercase + remove

---
## Summary & Key Takeaways

### What We Learned

| Technique | Strength | Weakness |
|---|---|---|
| **Data Cleaning** | Removes noise (HTML, URLs, special chars) | Can destroy signals (punctuation for sentiment, caps for emphasis) |
| **Whitespace Tokenization** | Simple and fast | Fails on contractions, hyphenation, punctuation |
| **NLTK Tokenizer** | Handles contractions (Penn Treebank rules) | Keeps hyphenated words as one token |
| **spaCy Tokenizer** | Splits hyphenation, provides POS/dep | Slower, splits more aggressively |
| **Stopword Removal** | Reduces vocabulary, focuses on content words | Removes negation words, flips meaning |
| **Porter Stemmer** | Fast, groups word forms aggressively | Produces non-words (over-stemming) |
| **Snowball Stemmer** | Slightly more conservative than Porter | Still produces non-words |
| **Lemmatizer (spaCy)** | Returns real dictionary words, uses context | Slower, requires POS tagging |

### Critical Tradeoffs to Remember

1. **Lowercasing** merges `US` (country) with `us` (pronoun) — lossy for NER tasks
2. **Stopword removal** deletes `not` — catastrophic for sentiment analysis
3. **Stemming** is fast but crude; **lemmatization** is accurate but slow
4. **No single pipeline works for all tasks** — always customize for your domain
